In [1]:
%%capture

import altair as alt
import gcsfs
import pandas as pd

from IPython.display import HTML, Markdown, display
#from update_vars import GCS_FILE_PATH, MONTH, PUBLIC_FILENAME, YEAR
from gtfs_curator_utils import magics

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

WIDTH = 300
HEIGHT = 150

In [2]:
# parameters cell for local
rtpa = "Metropolitan Transportation Commission"

In [3]:
%%capture_parameters
rtpa

{"rtpa": "Metropolitan Transportation Commission"}


# {rtpa}
Annual Ridership Trends

Download data from our **[public folder](https://console.cloud.google.com/storage/browser/calitp-publish-data-analysis)** by navigating to `ntd_annual_ridership` and selecting a file.

Transit operators/agencies that submit annual reports to NTD are included in this report. Reporters that were previously active reporters, but are currently not, may appear. This may result in Reporters showing zero or partial ridership data in the report.

If a Reporter, type of service, mode, or any combination of, is not a annual reporter or has not reported data since 2018, they will not appear in the report.

Examples:

* **Reporter A** is an annual reporter from 2019-2022, then became inactive and did not report for 2023. Reporter A's ridership data will be displayed for 2019-2022 only.
* **Reporter B** is an annual from 2000-2017, then became inactive and did not report for 2018. Reporter B will be named in the report, but will not display ridership data.
* **Reporter C** was an inactive reporter form 2015-2020, then became an active full reporter for 2021. Reporter C's ridership data will be displayed for 2021-present.


# need to set PUBLIC_FILENAME in update_vars
URL = "https://console.cloud.google.com/storage/" "browser/calitp-publish-data-analysis"

display(
    HTML(
        f"""
        <a href={URL}>
        Download the latest month of data: {PUBLIC_FILENAME}</a>
        """
    )
)

* [annual ridership query](https://github.com/tiffanychu90/curator/blob/use-new-ntd-tables/ntd/ntd_utils.py#L293)
   * includes `agency_status`, what is this? 

In [4]:
import B3_ntd_utils as ntd_utils

def merge_new_df_with_crosswalk(
    filename: str = "annual"
):
    """
    How can this function not repeat with each chapter?
    Can move it into Python script and create this
    """
    crosswalk = pd.read_parquet(
        f"{GCS_FILE_PATH}crosswalk2.parquet", 
        filesystem=gcsfs.GCSFileSystem(),
        columns = ["ntd_id_2022", "rtpa_name", "rtpa_name_split"]
    ).rename(columns = {"ntd_id_2022": "ntd_id"})

    df = pd.read_parquet(
        f"{GCS_FILE_PATH}{filename}.parquet",
        # should only certain columns be read in? now this table is much larger
        filesystem=gcsfs.GCSFileSystem(),
    ).merge(
        crosswalk,
        on = "ntd_id",
        how = "left"
    )

    if filename == "annual":
        # for annual, use rtpa_name_split
        df = df.assign(
            rtpa_name = df.apply(ntd_utils.extra_annual_rtpa_splitting, axis=1)
        ).rename(columns = {"unlinked_passenger_trips": "upt"})

    return df

def filter_to_rtpa(df: pd.DataFrame, one_rtpa: str):
    return df[df.rtpa_name == one_rtpa].reset_index(drop=True)

In [5]:
# read in data
df = merge_new_df_with_crosswalk("annual").pipe(filter_to_rtpa, rtpa)

In [6]:
# this is total upt since 2018, which is a parameter in the query
# might need to set this in update_vars, otherwise if it updates, we don't know and caption is wrong
# These counts are totally over counting, more than double, look into why
def proportion_of_upt_by_agency(df: pd.DataFrame):
    initial_agg = (
        df
        .groupby("source_agency")
        .agg(
            total_upt=("upt", "sum")
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by="total_upt", ascending=False)
    )
     # % total columns
    initial_agg["pct_of_total_upt"] = ((
        initial_agg["total_upt"] / initial_agg["total_upt"].sum()
    ) * 100).round(decimals=2)

    return initial_agg

In [7]:
# agg by agency, for pie chart
agency_agg_yr = df.pipe(proportion_of_upt_by_agency)
total_upt = agency_agg_yr.total_upt.sum()
agency_count = agency_agg_yr.source_agency.nunique()

## Report Totals

In [8]:
Markdown(
    f"""
Within {rtpa}:
- Number of Reporters: <b>{agency_count}</b>.
- Total Unlinked Passenger Trips since the beginning of this report: <b>{total_upt:,}</b>.
- Individual Reporters ridership breakdown:
"""
)


Within Metropolitan Transportation Commission:
- Number of Reporters: <b>23</b>.
- Total Unlinked Passenger Trips since the beginning of this report: <b>2,331,534,369</b>.
- Individual Reporters ridership breakdown:


**upt bar chart notes**
* this one has parameter for year
* why does this not use the make_bar_chart function?


In [9]:
def make_bar_chart_new(
    df: pd.DataFrame, x_col: str, y_col: str, color_col: str, tooltip_cols: list
):

    if y_col=="total_upt":
        y_sort="-y"
    else:
        y_sort=None
    
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(x_col, sort=None), 
            y=alt.Y(y_col, title=y_col, sort=y_sort), 
            color=alt.Color(
                color_col, title="", sort=None,
                scale=alt.Scale(
                    range=ntd_utils.CALITP_CATEGORY_BRIGHT_COLORS + 
                    ntd_utils.CALITP_CATEGORY_BOLD_COLORS),
            ),
            tooltip = tooltip_cols
        ).properties(width=WIDTH, height=HEIGHT)
        .interactive()
    )
    return chart

In [10]:
# can't layer encodings anymore
#https://stackoverflow.com/questions/75450068/typeerror-undefinedtype-object-is-not-callable
# maybe this bar chart can display what pie chart (isn't used anymore) and show another y-axis of percent

In [11]:
# simple bar chart for total agencies and UPT
total_upt_chart = make_bar_chart_new(
    agency_agg_yr,
    x_col="source_agency",
    y_col="total_upt",
    color_col="source_agency",
    tooltip_cols = ["source_agency", "total_upt", "pct_of_total_upt"]
).properties(title="Total Annual Unlinked Passenger Trips per Reporter in RTPA since 2018") # title needs parameter

total_upt_chart

alt.Chart(...)

In [12]:
def make_base_chart(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    color_col: str,
    tooltip_cols: list
) -> alt.Chart:
    # line and bar chart differ in how many years they pull?
    #df = df[df[y_col] > 0].dropna(subset=y_col)

    chart = (
        alt.Chart(df)
        #.mark_line(point=True)
        .encode(
            x=alt.X(x_col),    
            y=alt.Y(y_col, title=y_col),
            color=alt.Color(
                color_col,
                scale=alt.Scale(
                    range=ntd_utils.CALITP_CATEGORY_BRIGHT_COLORS + 
                    ntd_utils.CALITP_CATEGORY_BOLD_COLORS),
            ),
            tooltip=tooltip_cols
        ).properties(width=WIDTH, height=HEIGHT)
        .interactive()
    )

    return chart

## check all the aggregations

In [13]:
by_agency_long2 = ntd_utils.aggregate_by_agency(
    df, 
    previous_upt_col = "upt_prior_year", 
    time_cols = ["year"], 
    geography_cols = ["rtpa_name"]
)

by_mode_long2 = ntd_utils.aggregate_by_mode(
    df,
    previous_upt_col = "upt_prior_year",
    time_cols = ["year"],
    geography_cols = ["rtpa_name"]
)

by_tos_long2 = ntd_utils.aggregate_by_tos(
    df,
    previous_upt_col = "upt_prior_year", # this groupby uses type_of_service_full_name and type_of_service
    time_cols = ["year"],
    geography_cols = ["rtpa_name"]
)

by_reporter_type_long2 = ntd_utils.aggregate_by_reporter_type(
    df,
    previous_upt_col = "upt_prior_year",
    time_cols = ["year"],
    geography_cols = ["rtpa_name"]
)

## Agency

possibly the line chart and difference chart can be side by side?

In [14]:
make_base_chart(
    by_agency_long2,
    x_col = "year",
    y_col="upt",
    color_col="source_agency",
    tooltip_cols=["year", "upt", "source_agency", "rtpa_name"] 
).mark_line(point=True).facet(
    "source_agency", columns = 2, title=""
).resolve_scale(y="independent").properties(
    title="Total Annual Unlinked Passenger Trips by Reporter"
)

alt.FacetChart(...)

In [15]:
make_base_chart(
    by_agency_long2,
    x_col = "year:O",
    y_col="upt_change_1yr",
    color_col = "source_agency",
    tooltip_cols=["year", "upt_change_1yr", "source_agency", "rtpa_name"],
).mark_bar().facet(
    "source_agency", columns=2, title=""
).resolve_scale(y="independent").properties(title="Yearly Change in Unlinked Passenger Trips by Agency")

alt.FacetChart(...)

## Transit Mode

still need to use mode_full_name

In [16]:
make_base_chart(
    by_mode_long2,
    x_col = "year:O",
    y_col="upt_change_1yr",
    color_col = "mode",
    tooltip_cols=["year", "upt_change_1yr", "mode", "rtpa_name"],
).mark_line(point=True).facet(
    "mode", columns=2, title=""
).resolve_scale(y="independent").properties(title="Total Annual UPT by Mode")

alt.FacetChart(...)

In [17]:
make_base_chart(
    by_mode_long2,
    x_col = "year:O",
    y_col="upt_change_1yr",
    color_col = "mode",
    tooltip_cols=["year", "upt_change_1yr", "mode", "rtpa_name"],
).mark_bar().facet(
    "mode", columns=2, title=""
).resolve_scale(y="independent").properties(title="Yearly Change in UPT by Mode")


alt.FacetChart(...)

## Type of Service

In [18]:
by_tos_long2.dtypes

type_of_service     object
year                 Int64
rtpa_name           object
upt                  Int64
upt_prior_year       Int64
upt_change_1yr       Int64
pct_change_1_yr    Float64
dtype: object

In [19]:
make_base_chart(
    by_tos_long2,
    x_col = "year:O",
    y_col="upt",
    color_col="type_of_service",
    tooltip_cols=["year", "upt_change_1yr", "type_of_service", "rtpa_name"],
).mark_line(point=True).facet(
    "type_of_service", columns=2, title=""
).resolve_scale(y="independent").properties(title="Total Annual UPT by Type of Service")


make_base_chart(
    by_tos_long2,
    x_col = "year:O",
    y_col="upt_change_1yr",
    color_col = "type_of_service",
    tooltip_cols=["year", "upt_change_1yr", "type_of_service", "rtpa_name"],
).mark_bar().facet(
    "type_of_service", columns=2, title=""
).resolve_scale(y="independent").properties(title="Total Change in UPT by Type of Service")

alt.FacetChart(...)

## Reporter Type

In [20]:
make_base_chart(
    by_reporter_type_long2,
    x_col = "year:O",
    y_col="upt",
    color_col="reporter_type",
    tooltip_cols=["year", "upt_change_1yr", "reporter_type", "rtpa_name"],
).mark_line(point=True).facet(
    "reporter_type", columns=2, title=""
).resolve_scale(y="independent").properties(title="Total Annual UPT by NTD Reporter Type")


make_base_chart(
    by_reporter_type_long2,
    x_col = "year:O",
    y_col="upt_change_1yr",
    color_col = "reporter_type",
    tooltip_cols=["year", "upt_change_1yr", "reporter_type", "rtpa_name"],
).mark_bar().facet(
    "reporter_type", columns=2, title=""
).resolve_scale(y="independent").properties(title="Total Change in UPT by Reporter Type")

alt.FacetChart(...)

In [ ]:

def bar_chart_titles_by_grouping():
    """
    TODO: bar chart titles aren't consistent here, but they really can be suffixed with grouping
    """
    #by_agency: Yearly Change in Unlinked Passenger Trips by Agency
    #by_mode: Yearly Change in UPT by Mode
    #by_tos: Total Change in UPT by Type of Service
    #by_reporter_type: Total Annual UPT by NTD Reporter Type
    return

def total_upt_title(grouping: str):
    prefix = "Annual UPT by"
    {
        "reporter_type": "Reporter Type",
        "type_of_service":
    }
    if grouping=="reporter_type":
        return 
    grouping.title()
    return f"{prefix} {grouping}"

def change_upt_title():
    return